# SOM-Based K-Nearest Neighbor Algorithm for Network Intrusion Detection

## Project Overview

This notebook implements a **Network Intrusion Detection System (NIDS)** using:
- **Self-Organizing Maps (SOM)** for unsupervised learning of network traffic patterns
- **K-Nearest Neighbor (KNN)** distance scoring for anomaly detection
- **Bayesian Optimization** (hyperopt) for hyperparameter tuning

### Dataset
We use the [NSL-KDD](https://www.unb.ca/cic/datasets/nsl.html) dataset, which contains 41 network traffic features and 5 attack categories:

| Category | Label | Examples |
|----------|-------|----------|
| Normal   | 0     | Benign traffic |
| Probe    | 1     | ipsweep, nmap, portsweep, satan |
| R2L      | 2     | ftp_write, guess_passwd, phf |
| DoS      | 3     | neptune, smurf, back, teardrop |
| U2R      | 4     | buffer_overflow, rootkit, perl |

### Approach
1. Load and preprocess NSL-KDD data
2. Feature selection using ExtraTreesClassifier
3. Train SOM on benign traffic only
4. Use KNN distance from SOM nodes as anomaly score
5. Evaluate detection rates per attack category

## 1. Setup and Imports

Import all required libraries and project modules. We use `sys.path` to reference our refactored `src/` package.

In [ ]:
import sys
import os
import time

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy.stats import gaussian_kde
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.feature_selection import SelectFromModel
from sklearn.preprocessing import MinMaxScaler

# Add project root to path so we can import from src/
sys.path.insert(0, os.path.abspath('..'))

from src.som_anomaly_detector import MiniSom, AnomalyDetection
from src.preprocessing import (
    COLUMN_NAMES, ATTACK_CATEGORY_MAP, CATEGORY_NAMES,
    load_nsl_kdd, encode_attack_categories, preprocess_features,
)
from src.visualization import plot_som, plot_anomaly_density, get_anomalies

%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

## 2. Data Loading and Preprocessing

### 2.1 Load the NSL-KDD Dataset

The NSL-KDD dataset consists of:
- **KDDTrain+.csv** — 125,973 training samples
- **KDDtest.csv** — 22,544 test samples

Each sample has 41 features covering:
- **Basic connection features** (duration, protocol, service, flag)
- **Content features** (failed logins, root shell access)
- **Traffic features** (connection counts, error rates)
- **Host-based features** (destination host statistics)

In [ ]:
# Load and combine datasets
DATA_DIR = os.path.join('..', 'data')
full, n_train = load_nsl_kdd(
    os.path.join(DATA_DIR, 'KDDTrain+.csv'),
    os.path.join(DATA_DIR, 'KDDtest.csv'),
)

print(f"Training samples: {n_train:,}")
print(f"Test samples: {full.shape[0] - n_train:,}")
print(f"Total features: {full.shape[1] - 2}")  # minus outcome and score
full.head()

### 2.2 Encode Attack Categories

We map the 39 specific attack types into 5 categories (Normal, Probe, R2L, DoS, U2R) for classification.

In [ ]:
# Encode attack types into 5 categories
full['outcome'] = encode_attack_categories(full)
del full['score']

# Show class distribution
print("Class distribution:")
for cat_id, cat_name in CATEGORY_NAMES.items():
    count = (full['outcome'] == cat_id).sum()
    print(f"  {cat_name} ({cat_id}): {count:,} samples")

### 2.3 Feature Engineering

One-hot encode the categorical features (`protocol_type`, `service`, `flag`). This expands the feature space from 41 to ~120 features.

In [ ]:
# One-hot encoding of categorical features
full_encoded = pd.get_dummies(full, drop_first=True)
print(f"Features after one-hot encoding: {full_encoded.shape[1] - 1}")

# Split back into train and test
df_train = full_encoded[:n_train]
df_test = full_encoded[n_train:]

# Separate features by attack category for per-class evaluation
target_train = df_train['outcome']
target_test = df_test['outcome']

data_benign = df_train[df_train['outcome'] == 0].drop(columns=['outcome'])
data_train = df_train.drop(columns=['outcome'])

# Test set split by attack category
test_by_category = {}
for cat_id, cat_name in CATEGORY_NAMES.items():
    test_by_category[cat_name] = df_test[df_test['outcome'] == cat_id].drop(columns=['outcome'])
    print(f"  Test {cat_name}: {test_by_category[cat_name].shape[0]:,} samples")

data_test = df_test.drop(columns=['outcome'])

### 2.4 Feature Scaling

Apply Min-Max scaling to normalize all features to the [0, 1] range. This is important for SOM training since distance calculations are sensitive to feature scales.

In [ ]:
scaler = MinMaxScaler(feature_range=(0, 1))

data_train_scaled = scaler.fit_transform(data_train)
data_test_scaled = scaler.fit_transform(data_test)
data_benign_scaled = scaler.fit_transform(data_benign)

# Scale each test category separately
test_scaled = {}
for name, data in test_by_category.items():
    test_scaled[name] = scaler.fit_transform(data)

print(f"Scaled training shape: {data_train_scaled.shape}")
print(f"Scaled benign shape: {data_benign_scaled.shape}")

### 2.5 Feature Selection

Use **ExtraTreesClassifier** (tree-based feature importance) to select the most discriminative features. This reduces dimensionality and can improve SOM training efficiency.

In [ ]:
# Feature selection using ExtraTreesClassifier
clf = ExtraTreesClassifier(random_state=42)
clf.fit(data_train_scaled, target_train)
selector = SelectFromModel(clf, prefit=True)

# Transform all datasets
data_benign_new = selector.transform(data_benign_scaled)
data_train_new = selector.transform(data_train_scaled)
data_test_new = selector.transform(data_test_scaled)

test_new = {}
for name, data in test_scaled.items():
    test_new[name] = selector.transform(data)

print(f"Features after selection: {data_benign_new.shape[1]}")
print(f"Reduction: {data_train_scaled.shape[1]} -> {data_train_new.shape[1]} features")

## 3. SOM Hyperparameter Tuning

### Bayesian Optimization with Hyperopt

We use **Tree-structured Parzen Estimator (TPE)** to find optimal SOM hyperparameters:
- `sigma` — neighborhood spread (5–10)
- `learning_rate` — initial learning rate (0.05–5)
- `grid_size` — SOM grid dimensions (20–50)

The objective is to **minimize quantization error** on benign training data.

In [ ]:
from hyperopt import Trials, STATUS_OK, fmin, tpe, hp

# Define the search space
space = {
    'sigma': hp.uniform('sigma', 5, 10),
    'learning_rate': hp.uniform('learning_rate', 0.05, 5),
    'x': hp.uniform('x', 20, 50),
}

def som_objective(params):
    """Objective: minimize quantization error on benign data."""
    x = int(params['x'])
    som = MiniSom(
        x=x, y=x,
        input_len=data_benign_new.shape[1],
        sigma=params['sigma'],
        learning_rate=params['learning_rate'],
    )
    error = som.quantization_error(data_benign_new[:100])
    return {'loss': error, 'status': STATUS_OK}

trials = Trials()
best_som_params = fmin(
    fn=som_objective,
    space=space,
    algo=tpe.suggest,
    max_evals=1200,
    trials=trials,
)

print(f"\nBest SOM parameters: {best_som_params}")

## 4. SOM Training and Visualization

Train the SOM with the optimized hyperparameters on **benign traffic only**. The SOM learns the structure of normal network behavior. We then visualize how different attack types map onto the trained SOM using the U-matrix (distance map).

In [ ]:
# Train SOM with optimized parameters
start = time.time()

som = MiniSom(
    x=43, y=43,
    input_len=data_benign_new.shape[1],
    sigma=8.164432071013474,
    learning_rate=1.95477168894887272,
)
som.train_random(data_benign_new, 1000)

elapsed = time.time() - start
print(f"SOM training time: {elapsed:.2f} seconds")

# Visualize: plot first 1000 training samples on the SOM
n_plot = 1000
plot_som(som, data_train_new[:n_plot], target_train.values[:n_plot])

## 5. Anomaly Detection

### 5.1 Anomaly Detection Algorithm

The `AnomalyDetection` class combines SOM with KNN:

1. **Train** a Kohonen SOM on benign data
2. **Count** how many samples activate each SOM node (BMU counts)
3. **Prune** nodes with fewer activations than `min_bmu_count`
4. For each new sample, compute **mean KNN distance** to the remaining nodes
5. Samples far from all nodes → **high anomaly score**

### 5.2 Helper Functions

In [ ]:
def minimize_anomaly(benign_metrics, anomaly_metrics, alpha=3):
    """Compute misclassification rate (for hyperopt minimization)."""
    limit = np.mean(benign_metrics) + np.std(benign_metrics) * alpha
    outliers = np.argwhere(np.abs(anomaly_metrics) > limit)
    pct_anomaly = len(outliers) / len(anomaly_metrics)
    return 1 - pct_anomaly

# Training data = benign traffic, Evaluation data = test set
training = data_benign_new
evaluation = data_test_new

print(f"Training samples (benign): {training.shape[0]:,}")
print(f"Evaluation samples (test): {evaluation.shape[0]:,}")

### 5.3 Anomaly Detector Hyperparameter Tuning

Bayesian Optimization over 7 hyperparameters:
- `n_neighbors` — KNN neighbors for distance scoring
- `learning_rate`, `learning_decay` — SOM training parameters
- `initial_radius`, `radius_decay` — SOM neighborhood parameters
- `x` — SOM grid size
- `minNumberPerBmu` — pruning threshold

In [ ]:
from hyperopt import Trials, STATUS_OK, fmin, tpe, hp

space = {
    'n_neighbors': hp.uniform('n_neighbors', 1, 6),
    'learning_rate': hp.uniform('learning_rate', 0.005, 10),
    'learning_decay': hp.uniform('learning_decay', 0.00001, 0.1),
    'initial_radius': hp.uniform('initial_radius', 1, 10),
    'radius_decay': hp.uniform('radius_decay', 0.00001, 0.1),
    'x': hp.uniform('x', 10, 50),
    'minNumberPerBmu': hp.uniform('minNumberPerBmu', 0, 10),
}

def anomaly_objective(params):
    """Objective: minimize misclassification in anomaly detection."""
    detector = AnomalyDetection(
        shape=(int(params['x']), int(params['x'])),
        input_size=training.shape[1],
        learning_rate=params['learning_rate'],
        learning_decay=params['learning_decay'],
        initial_radius=int(params['initial_radius']),
        radius_decay=params['radius_decay'],
        min_bmu_count=int(params['minNumberPerBmu']),
        n_neighbors=int(params['n_neighbors']),
    )
    detector.fit(training, 5000)
    a_metrics = detector.evaluate(evaluation)
    b_metrics = detector.evaluate(training)
    val = minimize_anomaly(b_metrics, a_metrics, alpha=3)
    return {'loss': val, 'status': STATUS_OK}

trials = Trials()
best_anomaly_params = fmin(
    fn=anomaly_objective,
    space=space,
    algo=tpe.suggest,
    max_evals=100,
    trials=trials,
)

print(f"\nBest anomaly detector parameters: {best_anomaly_params}")

### 5.4 Train Final Anomaly Detector

Using the optimized hyperparameters, train the final anomaly detection model with 10,000 iterations for thorough convergence.

In [ ]:
start = time.time()

anomaly_detector = AnomalyDetection(
    shape=(20, 20),
    input_size=training.shape[1],
    learning_rate=9.800275263995829,
    learning_decay=0.0029030131893884627,
    initial_radius=4.13960012604465,
    radius_decay=0.0012626284950839894,
    min_bmu_count=7.304609930044314,
    n_neighbors=2,
)

anomaly_detector.fit(training, 10000)

elapsed = time.time() - start
print(f"Anomaly detector training time: {elapsed:.2f} seconds")

## 6. Evaluation and Results

### 6.1 Compute Anomaly Scores and Threshold

The threshold is set at **mean + 3 * std** of the benign scores. Samples exceeding this threshold are flagged as anomalous.

In [ ]:
# Compute anomaly scores
benign_metrics = anomaly_detector.evaluate(data_benign_new)

alpha = 3
sd_benign = np.std(benign_metrics)
mean_benign = np.mean(benign_metrics)
threshold_sd = mean_benign + alpha * sd_benign
threshold_pct = np.percentile(benign_metrics, 99.7)

print(f"Benign score mean: {mean_benign:.4f}")
print(f"Benign score std: {sd_benign:.4f}")
print(f"Threshold (mean + 3*std): {threshold_sd:.4f}")
print(f"Threshold (99.7th percentile): {threshold_pct:.4f}")

### 6.2 Visualize Detection Results

Density plots show how benign vs. attack traffic distributes relative to the decision threshold. Good separation means higher detection rates.

In [ ]:
# Evaluate all test data
anomaly_metrics = anomaly_detector.evaluate(evaluation)

# Benign distribution
plot_anomaly_density(benign_metrics, benign_metrics, threshold_sd, title="Benign")

# All anomalies
plot_anomaly_density(benign_metrics, anomaly_metrics, threshold_sd, title="All Anomalies")

# Per-category analysis
for name in ['Probe', 'R2L', 'DoS', 'U2R']:
    if name in test_new and test_new[name].shape[0] > 0:
        cat_metrics = anomaly_detector.evaluate(test_new[name])
        plot_anomaly_density(benign_metrics, cat_metrics, threshold_sd, title=name)

### 6.3 Detection Rates Summary

Final detection rates per attack category using the standard deviation threshold.

In [ ]:
print("=" * 50)
print("DETECTION RESULTS (alpha = 3)")
print("=" * 50)

# Benign false positive rate
print("\nBenign (false positives):")
get_anomalies(benign_metrics, benign_metrics, alpha, return_outliers=False)

# All anomalies
print("\nAll anomalies:")
get_anomalies(benign_metrics, anomaly_metrics, alpha, return_outliers=False)

# Per category
for name in ['Probe', 'R2L', 'DoS', 'U2R']:
    if name in test_new and test_new[name].shape[0] > 0:
        print(f"\n{name}:")
        cat_metrics = anomaly_detector.evaluate(test_new[name])
        get_anomalies(benign_metrics, cat_metrics, alpha, return_outliers=False)

## 7. Conclusions

### Key Findings

| Attack Type | Detection Rate | Notes |
|-------------|---------------|-------|
| Benign (FP) | ~1.7% | Low false positive rate |
| Probe | ~90.1% | Excellent detection |
| DoS | ~85.7% | Strong detection |
| U2R | ~100% | Perfect detection |
| R2L | ~5.4% | Poor — needs improvement |
| All Anomalies | ~39.7% | Weighted by class distribution |

### Strengths
- **Unsupervised approach**: Only needs benign traffic for training
- **Excellent U2R and Probe detection**: SOM captures structural differences well
- **Low false positive rate**: Important for practical deployment

### Limitations
- **R2L detection is weak**: R2L attacks mimic normal traffic patterns closely
- **Overall detection rate skewed**: Dominated by DoS samples in the dataset

### Future Work
- Combine with supervised classifier for R2L detection
- Explore ensemble methods (SOM + ANN)
- Test on more recent datasets (CICIDS2017, CSE-CIC-IDS2018)